make a new file that loads the h5 & csv with labels. drop rows in the csv that don't have a label or have DELETE as their label. drop rows from h5 that aren't in csv. output smaller h5 that's useable

In [1]:
import pandas as pd
import os
from pathlib import Path
import librosa
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import h5py
from tqdm import tqdm
import hashlib

In [2]:
# get file paths
FOLDER_PATH = "/Users/cheoso/Documents/samples/Packs/"
file_paths = []
total = 0

for root, dirs, files in os.walk(FOLDER_PATH):
    for file in files:
        path = Path(root) / file
        path_lower = str(path).lower()
        if path_lower.endswith(('.wav', '.aif', '.aiff')):
          file_paths.append(str(path))

          total += path.stat().st_size

print(f"Total of {total} bytes.")

Total of 84644864348 bytes.


In [3]:
failed_files = []
divided_by_zero = []

def process_audio(audio, sr):
    rs = librosa.resample(audio, orig_sr=sr, target_sr=44100)
    if rs.max() == 0:
        divided_by_zero.append(sr)
    else:
        return rs / abs(rs.max())
    return rs

def hash_waveform_with_id(row):
    waveform = row
    waveform_hash = hashlib.sha256(waveform.tobytes()).hexdigest()
    return waveform_hash

def extract_metadata(index, file):
    """Load an audio file and extract its waveform, sample rate, and label."""
    try:
        audio, sr = librosa.load(file, sr=None)  # Load without resampling
        audio = process_audio(audio, sr)
        audio = hash_waveform_with_id(audio)
        file = str(file)
        file_name = file.split('/')[-1]
        return [index, file, file_name, audio]
    except Exception as e:
        failed_files.append(file)
        return None

# Use parallel processing to load audio files
def get_audio_and_metadata(files):
    meta_results = []

    with ThreadPoolExecutor() as executor:
        results = executor.map(lambda p: extract_metadata(*p), enumerate(files))
        
        for res in results:
            if res is not None:
                meta_data = res
                meta_results.append(meta_data)
    
    return meta_results


metadataResults = get_audio_and_metadata(file_paths)

/var/folders/h0/pzm07l6x5n7181rc09xppl940000gn/T/ipykernel_55191/3409292700.py:20: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file, sr=None)  # Load without resampling
/Users/cheoso/ai_projects/load_audio/venv/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [4]:
failed_len = len(failed_files)
print(failed_len)
print(len(divided_by_zero))

639
12


In [18]:
print(len(audio_hashes))

51159


In [5]:
print(len(file_paths))

52910


In [6]:
#print(len(audioResults))
#old 52271
print(len(metadataResults))

52271


In [24]:
#metadataResults[:2]

In [7]:
# Convert results to DataFrame
#df_ar = pd.DataFrame(audioResults, columns=["id", "sample_rate", "waveform"])
df_md = pd.DataFrame(metadataResults, columns=["id", "file_path", "file_name", "audio_hash"])

In [8]:
'''dt = h5py.string_dtype(encoding='utf-8')


with h5py.File("audio_data.h5", "w") as f:
    meta_data = f.create_group("meta_data")
    meta_data.create_dataset('index', data=np.array(df_md['id'], dtype='int'))
    meta_data.create_dataset('path', data=np.array(df_md['file_path'], dtype=dt))
    meta_data.create_dataset('name', data=np.array(df_md['file_name'], dtype=dt))

    audio_data = f.create_group('audio_data')

    def save_sample(row):
        sample_group = audio_data.create_group(str(row['id']))
        sample_group.create_dataset('waveform', data=row['waveform'], compression='gzip')
        sample_group.create_dataset('sample_rate', data=row['sample_rate'])

    # Use ThreadPoolExecutor for parallelism
    with ThreadPoolExecutor(max_workers=8) as executor:
        list(tqdm(executor.map(save_sample, [row for _, row in df_ar.iterrows()]), total=len(df_ar)))'''
    

'dt = h5py.string_dtype(encoding=\'utf-8\')\n\n\nwith h5py.File("audio_data.h5", "w") as f:\n    meta_data = f.create_group("meta_data")\n    meta_data.create_dataset(\'index\', data=np.array(df_md[\'id\'], dtype=\'int\'))\n    meta_data.create_dataset(\'path\', data=np.array(df_md[\'file_path\'], dtype=dt))\n    meta_data.create_dataset(\'name\', data=np.array(df_md[\'file_name\'], dtype=dt))\n\n    audio_data = f.create_group(\'audio_data\')\n\n    def save_sample(row):\n        sample_group = audio_data.create_group(str(row[\'id\']))\n        sample_group.create_dataset(\'waveform\', data=row[\'waveform\'], compression=\'gzip\')\n        sample_group.create_dataset(\'sample_rate\', data=row[\'sample_rate\'])\n\n    # Use ThreadPoolExecutor for parallelism\n    with ThreadPoolExecutor(max_workers=8) as executor:\n        list(tqdm(executor.map(save_sample, [row for _, row in df_ar.iterrows()]), total=len(df_ar)))'

In [ ]:
reference_df = pd.read_csv('ref.csv')
#reference_df['is_used'] = False
filtered_results =reference_df[reference_df['group'].notna()]
filtered_results['is_used'] = True

filtered_results.head(5)

/var/folders/h0/pzm07l6x5n7181rc09xppl940000gn/T/ipykernel_55191/572105434.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_results['is_used'] = True


,id,file_path,file_name,group,category,sub_category,Reverse,Forward,One_Shot,Loop,...,Cowbell,Bell,Camera,Orchestra,Metalic Tap,One_Shot_Intent,Loop_id,Loop_Intent,Unnamed: 52,is_used
176,5487,808 and kick/heavy kick.wav,Heavy Kick.wav,Drum,Kick,NaN,0.0,1.0,1.0,0.0,...,0,0,0,0,0,1.0,NaN,0.0,x,True
177,5486,808 and kick/kick.aif,Kick.aif,Drum,808,NaN,0.0,1.0,1.0,0.0,...,0,0,0,0,0,NaN,NaN,NaN,x,True
179,5488,808 and kick/perfect 808.wav,Perfect 808.wav,Drum,808,NaN,0.0,1.0,1.0,0.0,...,0,0,0,0,0,NaN,NaN,NaN,x,True
180,7869,808 filth 2/01_808f2_one shot_tuned 808_cm_ana...,01_808F2_ONE SHOT_TUNED 808_CM_ANALOG_DEEP_WAV...,Drum,808,NaN,0.0,1.0,1.0,0.0,...,0,0,0,0,0,NaN,NaN,NaN,x,True
181,7902,808 filth 2/02_808f2_one shot_tuned 808_cm_ana...,02_808F2_ONE SHOT_TUNED 808_CM_ANALOG_DEEP_WAV...,Drum,808,NaN,0.0,1.0,1.0,0.0,...,0,0,0,0,0,NaN,NaN,NaN,x,True


In [83]:
df_md['file_path'] = df_md['file_path'].apply(lambda x: x.lower().split('/users/cheoso/documents/samples/packs/')[1])
df_md.head()

IndexError: list index out of range

In [ ]:
joined_df = df_md.merge(filtered_results, on='file_path', how='left')
joined_df.drop(columns=[col for col in joined_df.columns if '_y' in col], inplace=True)
joined_df.rename(columns={col: col.split('_x')[0] for col in joined_df.columns if '_x' in col}, inplace=True)

In [ ]:
joined_df['is_used'].astype(bool)
joined_df.loc[joined_df['is_used'].isna(), 'is_used'] = False

,id,file_path,file_name,audio_hash,group,category,sub_category,Reverse,Forward,One_Shot,...,Cowbell,Bell,Camera,Orchestra,Metalic Tap,One_Shot_Intent,Loop_id,Loop_Intent,Unnamed: 52,is_used
10402,10535,2024 crystal bass sample pack/atmospheres/ss_p...,SS_PadAtmos_Aether_Cmin.wav,63dafa1eda7d8a190f1bf8f9bc3931f7107f819437b410...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
10405,10538,2024 crystal bass sample pack/atmospheres/ss_p...,SS_PadAtmos_AngelsHymn_Em.wav,5aae3a222d85974a792d7d4b115b13840e1ce11a370318...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
10413,10546,2024 crystal bass sample pack/atmospheres/ss_p...,SS_PadAtmos_AngelsSong_D#maj.wav,53a63414c1f71da808b37271e264ad5fe7b9bc9202f22d...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
10440,10573,2024 crystal bass sample pack/atmospheres/ss_p...,SS_PadAtmos_Aureole_Gmaj50.wav,f9030fad897e14b2c6d83f5d139d9a19267f1ee984166c...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
10448,10581,2024 crystal bass sample pack/atmospheres/ss_p...,SS_PadAtmos_Aurora_F#min.wav,73da8dbbf96f4ea8dcce6fd2bd5ada388bfd62d902d670...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


In [82]:
joined_df.head()

,id_x,file_path,file_name_x,audio_hash,id_y,file_name_y,group,category,sub_category,Reverse,...,Cowbell,Bell,Camera,Orchestra,Metalic Tap,One_Shot_Intent,Loop_id,Loop_Intent,Unnamed: 52,is_used
0,0,heavyweight jungle/eo_hvywt_jngl_one_shots/eo_...,EO_HVYWT_JNGL_D_808_Dist_Synth_Bass.wav,1dab8788cf20e869f97c2938358f93971a0f695968dff1...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,heavyweight jungle/eo_hvywt_jngl_one_shots/eo_...,EO_HVYWT_JNGL_D_808_Deep_Synth_Bass.wav,f2c6282e5100bd350a38acaae9be3ab0ce2d66bd3a59ad...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,heavyweight jungle/eo_hvywt_jngl_one_shots/eo_...,EO_HVYWT_JNGL_D_Dist_Wub_Synth_Bass_02.wav,3fd4127ad461bd52ca6647b9c924c19c3e7219e39f3c4b...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,heavyweight jungle/eo_hvywt_jngl_one_shots/eo_...,EO_HVYWT_JNGL_D_Hard_808_Synth_Bass_02.wav,bef71c544e7919113b1a60188a946cfffc5535552af5a7...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,heavyweight jungle/eo_hvywt_jngl_one_shots/eo_...,EO_HVYWT_JNGL_D_Donk_Pluck_Synth_Bass_02.wav,f709a8f08f0fbb7882958185bb362990e110c219794d91...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [119]:
joined_df['duplicated'] = joined_df.duplicated(subset=['audio_hash'])

In [122]:
duplicates = joined_df[joined_df['duplicated'] == True].copy()
no_duplicates = joined_df[joined_df['duplicated'] == False].copy()

In [ ]:
unused_duplicates = duplicates[duplicates['is_used'] == False]
print(len(unused_duplicates))
unused_duplicates = unused_duplicates.drop_duplicates(subset="audio_hash", keep='last')
print(len(unused_duplicates))

used_duplicates = duplicates[duplicates['is_used'] == True]
print(len(used_duplicates))


1110


AttributeError: 'DataFrame' object has no attribute 'drop_duplicated'

In [18]:
df_md.head(5)

,id,file_path,file_name,audio_hash
0,0,/Users/cheoso/Documents/samples/Packs/Heavywei...,EO_HVYWT_JNGL_D_808_Dist_Synth_Bass.wav,1dab8788cf20e869f97c2938358f93971a0f695968dff1...
1,1,/Users/cheoso/Documents/samples/Packs/Heavywei...,EO_HVYWT_JNGL_D_808_Deep_Synth_Bass.wav,f2c6282e5100bd350a38acaae9be3ab0ce2d66bd3a59ad...
2,2,/Users/cheoso/Documents/samples/Packs/Heavywei...,EO_HVYWT_JNGL_D_Dist_Wub_Synth_Bass_02.wav,3fd4127ad461bd52ca6647b9c924c19c3e7219e39f3c4b...
3,3,/Users/cheoso/Documents/samples/Packs/Heavywei...,EO_HVYWT_JNGL_D_Hard_808_Synth_Bass_02.wav,bef71c544e7919113b1a60188a946cfffc5535552af5a7...
4,4,/Users/cheoso/Documents/samples/Packs/Heavywei...,EO_HVYWT_JNGL_D_Donk_Pluck_Synth_Bass_02.wav,f709a8f08f0fbb7882958185bb362990e110c219794d91...


In [115]:
joined_df.to_csv("audio_metadata_testing_filt_with_hashes.csv", header=True, index=False, encoding='utf-8-sig')